In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Paper formal baseline worker

Fixed method: `tree_ring`. This notebook only checks out the frozen producer and launches the resumable runner. It never changes thresholds, rosters, attacks, or denominators.


In [ ]:
import json, os, pathlib, subprocess, sys
from google.colab import userdata

REPO = 'https://github.com/RICHAAARC/CEG-WM.git'
EXPECTED_EXACT = '__BASELINE_FORMAL_PRODUCER_EXACT__'
METHOD = 'tree_ring'
JOB_ID = 'paper-baseline-treering-v1'
checkout = pathlib.Path('/content/cegwm-baseline-formal')
runtime_root = pathlib.Path('/content/cegwm-baseline-runtime') / METHOD
drive_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/PaperFormal-V1/baselines')

subprocess.run(['git', 'clone', REPO, str(checkout)], check=True)
subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', EXPECTED_EXACT], check=True)
head = subprocess.run(['git', '-C', str(checkout), 'rev-parse', 'HEAD'], check=True, capture_output=True, text=True).stdout.strip()
dirty = subprocess.run(['git', '-C', str(checkout), 'status', '--porcelain'], check=True, capture_output=True, text=True).stdout.strip()
assert head == EXPECTED_EXACT and not dirty
subprocess.run([sys.executable, '-m', 'pip', 'install', 'diffusers<0.40', 'transformers', 'accelerate', 'lpips', 'torchmetrics'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', str(checkout)], check=True)
child_env = dict(os.environ)
child_env['HF_TOKEN'] = userdata.get('HF_TOKEN') or ''
assert child_env['HF_TOKEN']
command = [sys.executable, '-m', 'experiments.run_paper_baseline_worker', '--method', METHOD, '--job-id', JOB_ID, '--expected-exact', EXPECTED_EXACT, '--drive-root', str(drive_root), '--runtime-root', str(runtime_root)]
subprocess.run(command, cwd=checkout, env=child_env, check=True)
final_path = drive_root / JOB_ID / 'method_final.json'
public = json.loads(final_path.read_text(encoding='utf-8'))
print({'method_id': public['method_id'], 'status': public['status'], 'result_package_produced': public['result_package_produced']})
